# A1. Model parameterization

As a *differentiable* modeling framework, AxonML allows users to define
model parameters as trainable tensors. This enables the use of gradient-based
optimization techniques to fit model parameters to experimental data and solve other optimization problems. This, however, introduces the complexity of facilitating complex parameter relationships, such as spatial gradients or regions with shared parameters. AxonML addresses this challenge by providing a flexible parameterization system that allows users to define parameters with custom spatial dependencies and constraints.

The core mixin in AxonML's parameterization system is the {py:class}`axonml.models.parametric.Parameterized` class, and the simpler {py:class}`axonml.models.parametric.SimpleParameterized` from which it inherits. All models and mechanisms in AxonML automatically inherit from these classes. These classes provide the foundational functionality for defining and managing model parameters. They allow users to specify parameters as tensors, define their spatial dependencies, and set constraints or relationships between parameters.

## Overview of parameterization backend


Parameters in AxonML can be one of three types:
- **Global parameters**: These are single values that apply uniformly across the entire model. They are defined as scalar tensors and can be used for parameters that do not vary spatially, such as global scaling factors or offsets. Use {py:meth}`axonml.models.parametric.Parameterized.GLOBAL` to define global parameters, and the positve-only variant {py:meth}`axonml.models.parametric.Parameterized.GLOBALP` if the parameter must be positive.

- **Range parameters**: These parameters vary spatially across the model domain. They are defined as tensors with shapes that match the model's spatial dimensions. Range parameters are useful for modeling spatially varying properties, such as concentration gradients or spatially dependent reaction rates. Use {py:meth}`axonml.models.parametric.Parameterized.RANGE` to define range parameters, and the positve-only variant {py:meth}`axonml.models.parametric.Parameterized.RANGEP` if the parameter must be positive.

- **Generic parameters**: These are user-defined parameters that can be added to a model as needed. They assume whatever shape the user defines (e.g., scalar, vector, matrix). Generic parameters provide flexibility for users to define custom parameters that may not fit into the global or range categories. Use {py:meth}`axonml.models.parametric.Parameterized.PARAMETER` to define generic parameters, and the positve-only variant {py:meth}`axonml.models.parametric.Parameterized.PARAMETERP` if the parameter must be positive.
Generic parameters can be defined using dictionaries to group related parameters together. This allows for organized management of multiple parameters, especially when they share similar characteristics or constraints.

:::{important}
For positive generic parameters defined using `superclass.PARAMETERP(parameter_name=value)`, use `parameterized_instance.evaluate(parameter_name)` to retrieve the actual parameter value. This ensures that the positivity constraint is respected. Direct access to `parameterized_instance.parameter_name` will return the underlying `PositiveParam` object, which cannot be used directly in computations. For non-positive generic parameters defined using `superclass.PARAMETER(parameter_name=value)`, direct access to `parameterized_instance.parameter_name` is sufficient. For GLOBALP and RANGEP parameters, direct access is also sufficient as they return the evaluated positive values in pre-populated buffers (discussed below).
:::

### Parameter vs buffer distinction